<a href="https://colab.research.google.com/github/saitejakchintaginjal/AIML-Projects/blob/main/medical_assitant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

saitejakchintaginjal_medical_diagnosis_manual_path = kagglehub.dataset_download('saitejakchintaginjal/medical-diagnosis-manual')

print('Data source import complete.')


## Business Context

Healthcare professionals often need to search through large volumes of medical information while making time-sensitive decisions. A RAG-based medical assistant can help retrieve relevant knowledge from trusted medical references and present it in a concise question-answer format.

## Objective

The objective of this project is to develop a Retrieval-Augmented Generation based medical assistant using a medical reference manual. The system combines semantic retrieval with a large language model to answer medical questions while improving the relevance and grounding of generated responses.

## Installation section

**Environment Setup**

The required libraries are installed for loading the GGUF language model, processing PDF documents, generating embeddings, creating the Chroma vector database, and building the RAG pipeline. CUDA-enabled llama-cpp-python is used to take advantage of GPU acceleration.

In [ ]:
# Install CUDA-enabled llama-cpp-python
!pip uninstall -y llama-cpp-python

!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    --force-reinstall --no-cache-dir

In [ ]:
import llama_cpp
from llama_cpp import Llama

print("llama-cpp-python imported successfully")
print("Version:", llama_cpp.__version__)
print("GPU offload supported:", llama_cpp.llama_supports_gpu_offload())

In [ ]:
!pip install -q \
    huggingface_hub \
    pandas \
    tiktoken \
    pymupdf \
    langchain \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    sentence-transformers==5.4.1 \
    transformers==4.57.1 \
    accelerate \
    -q

## Import Required Libraries

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

/tmp/ipykernel_145/1188750671.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [ ]:
import transformers
import sentence_transformers
import torch

print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Transformers: 4.57.1
Sentence Transformers: 5.4.1
CUDA: True
GPU: Tesla T4


# Question Answering using LLM

### Load the Hugging Face LLM

In [ ]:
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_file = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name,
    filename=model_file
)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=-1,
    verbose=False
)

### Create the Response Generation Function

### Validate the Loaded LLM

Before generating the project responses, a simple test query is used to confirm that the model is loaded correctly and can generate text successfully.

In [ ]:
response = llm(
    "What is hypertension? Answer in one short sentence.",
    max_tokens=100,
    temperature=0.1,
    top_p=0.9,
    top_k=40,
    stop=["</s>"]
)

print(response["choices"][0]["text"].strip())

Hypertension, also known as high blood pressure, is a chronic medical condition in which the force of blood against the walls of the arteries is consistently too high.


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/saitejakchintaginjal/medical-diagnosis-manual/medical_diagnosis_manual.pdf


In [ ]:
# Response Generation Function

def generate_response(
    question,
    temperature=0,
    top_p=0.95,
    top_k=50,
    max_tokens=512
):

    prompt = f"""[INST]
You are a medical knowledge assistant.

Answer the following question clearly and concisely.

Question:
{question}

Provide the answer:
[/INST]"""

    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    return response["choices"][0]["text"].strip()

### Define the Five Questions

In [ ]:
#Create one list so that we can reuse the questions throughout the entire project:

questions = [
    """What is the protocol for managing sepsis in a critical care unit?""",

    """What are the common symptoms of appendicitis, and can it be cured via medicine?
    If not, what surgical procedure should be followed to treat it?""",

    """What are the effective treatments or solutions for addressing sudden patchy hair loss,
    commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?""",

    """What treatments are recommended for a person who has sustained a physical injury
    to brain tissue, resulting in temporary or permanent impairment of brain function?""",

    """What are the necessary precautions and treatment steps for a person who has fractured
    their leg during a hiking trip, and what should be considered for their care and recovery?"""
]

### Generate Answers


In [ ]:
# Now apply our function to every question:

responses = []

for i, question in enumerate(questions, start=1):

    answer = generate_response(question)

    responses.append(answer)

    print(f"\n{'='*80}")
    print(f"QUESTION {i}")
    print(f"{'='*80}")
    print(question)

    print(f"\nANSWER {i}")
    print(f"{'-'*80}")
    print(answer)


QUESTION 1
What is the protocol for managing sepsis in a critical care unit?

ANSWER 1
--------------------------------------------------------------------------------
The protocol for managing sepsis in a critical care unit involves the following steps:

1. Early recognition and diagnosis: Identify sepsis suspects based on clinical suspicion, and perform quick diagnostic tests such as blood cultures and lactate levels.

2. Immediate fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and organ function. The goal is to achieve a mean arterial pressure (MAP) of 65 mmHg or higher and a central venous oxygen saturation (ScvO2) of 70% or higher.

3. Antibiotic administration: Start broad-spectrum antibiotics as soon as possible based on the suspected infection source and local microbial resistance patterns.

4. Source control: Identify and address the source of infection, such as removing an infected catheter or draining an abscess.

5. Vasopressor sup

## Observations


* The LLM was able to generate responses for all five medical questions and provided structured, point-wise explanations for each condition.

* For **sepsis**, the model provided a broad overview covering early recognition, fluid resuscitation, antibiotics, source control, vasopressors, corticosteroids, respiratory support, renal replacement therapy, monitoring, and multidisciplinary care. However, the response was quite lengthy and appears to be truncated toward the end.

* For **appendicitis**, the response clearly identified common symptoms such as abdominal pain, nausea, vomiting, fever, and appetite loss. It also identified **appendectomy** as the primary surgical treatment and mentioned both laparoscopic and open approaches.

* For **patchy hair loss**, the model identified **alopecia areata** and discussed possible causes and treatment options. However, some treatment statements are broad and would require validation against authoritative medical sources before being used for clinical decision-making.

* For **brain injury**, the model covered multiple aspects of management, including emergency stabilization, surgery, medications, rehabilitation, assistive devices, and lifestyle modifications. This demonstrates the LLM's ability to provide a broad treatment overview.

* For the **leg fracture**, the response covered immediate precautions, immobilization, seeking medical assistance, pain management, monitoring, transportation, rehabilitation, follow-up care, and prevention of future injuries.

* Overall, the standalone LLM provides **broad and well-structured medical information**, but the responses are generated from the model's pretrained knowledge and are **not grounded in the medical manuals provided for this project**.

* The responses also contain varying levels of detail and may include generalized or potentially outdated information. Therefore, the standalone LLM serves as a **baseline** and highlights the need for prompt engineering and RAG to improve the specificity, consistency, and grounding of the answers.

* The next stage will introduce **prompt engineering and parameter tuning** to determine whether the quality and consistency of the responses can be improved before incorporating the medical knowledge base through RAG.


# Question Answering using LLM with Prompt Engineering

### Why Prompt Engineering?

The baseline model was given a relatively simple instruction. In this section, we improve the prompt so that the model:

* Acts as a medical knowledge assistant.
* Provides structured and concise answers.
* Directly addresses all parts of the question.
* Avoids unnecessary information.
* Clearly indicates when information is uncertain.
* Does not make unsupported claims.


### Improved Prompt         

In [ ]:
# Function definition with improved prompt

def generate_prompt_engineered_response(
    question,
    temperature=0,
    top_p=0.95,
    top_k=50,
    max_tokens=512
):

    prompt = f"""[INST]
You are an expert medical knowledge assistant.

Your task is to answer the user's medical question accurately,
clearly, and in a structured manner.

Instructions:
1. Directly answer all parts of the question.
2. Provide the important symptoms, causes, treatments, or
   management steps relevant to the question.
3. Use clear and concise medical terminology.
4. Organize the response using numbered points or bullet points
   where appropriate.
5. Do not provide information that is unrelated to the question.
6. If the question requires medical evaluation, clearly state
   that professional medical assessment is necessary.
7. Do not invent facts or unsupported treatment recommendations.

Question:
{question}

Answer:
[/INST]"""

    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    return response["choices"][0]["text"].strip()

### Parameter Tuning

In [ ]:
# Rather than changing parameters randomly, we'll use 5 controlled combinations

parameter_combinations = [
    {
        "experiment": "Combination 1",
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "Combination 2",
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "Combination 3",
        "temperature": 0.5,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "Combination 4",
        "temperature": 0.0,
        "top_p": 0.90,
        "top_k": 40
    },
    {
        "experiment": "Combination 5",
        "temperature": 0.2,
        "top_p": 0.90,
        "top_k": 40
    }
]

### Run the Five Combinations

In [ ]:
# We need to answer all five questions for each combination.

prompt_engineering_results = []

for config in parameter_combinations:

    print("\n" + "="*100)
    print(config["experiment"])
    print("="*100)

    for i, question in enumerate(questions, start=1):

        answer = generate_prompt_engineered_response(
            question=question,
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"]
        )

        prompt_engineering_results.append({
            "Experiment": config["experiment"],
            "Question": f"Question {i}",
            "Temperature": config["temperature"],
            "Top_P": config["top_p"],
            "Top_K": config["top_k"],
            "Answer": answer
        })

        print(f"\nQuestion {i}:")
        print(question)

        print("\nAnswer:")
        print(answer)


Combination 1

Question 1:
What is the protocol for managing sepsis in a critical care unit?

Answer:
1. Recognition and Diagnosis:
   * Suspect sepsis in any patient with suspected or confirmed infection, and a suspected or documented inflammatory response.
   * Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to identify patients with sepsis and septic shock.

2. Immediate Interventions:
   * High-flow oxygen via a non-rebreather mask or endotracheal tube.
   * Intravenous access (minimum of two large-bore catheters).
   * Administer broad-spectrum antibiotics as soon as possible.
   * Initiate fluid resuscitation with crystalloids to maintain adequate tissue perfusion.

3. Hemodynamic Support:
   * Monitor and maintain mean arterial pressure (MAP) ≥65 mmHg.
   * Use vasopressors to maintain MAP if needed.
   * Consider the use of inotropes to improve cardiac output.

4. Fluid Management:
   * Monitor urine output, central venous pressure, and pulmonary arte

### Display Results as a DataFrame

In [ ]:
prompt_engineering_df = pd.DataFrame(prompt_engineering_results)

prompt_engineering_df[
    [
        "Experiment",
        "Question",
        "Temperature",
        "Top_P",
        "Top_K",
        "Answer"
    ]
]

,Experiment,Question,Temperature,Top_P,Top_K,Answer
0,Combination 1,Question 1,0.0,0.95,50,1. Recognition and Diagnosis:\n * Suspect se...
1,Combination 1,Question 2,0.0,0.95,50,1. Common symptoms of appendicitis include:\n ...
2,Combination 1,Question 3,0.0,0.95,50,1. Possible Causes:\n a. Androgenetic Alopec...
3,Combination 1,Question 4,0.0,0.95,50,1. The treatment for a person with a brain inj...
4,Combination 1,Question 5,0.0,0.95,50,1. Immediate Precautions:\n a. Ensure the pe...
5,Combination 2,Question 1,0.2,0.95,50,1. Recognition and Diagnosis:\n * Identify s...
6,Combination 2,Question 2,0.2,0.95,50,1. Common symptoms of appendicitis include:\n ...
7,Combination 2,Question 3,0.2,0.95,50,1. Possible Causes:\n a. Androgenetic Alopec...
8,Combination 2,Question 4,0.2,0.95,50,1. The treatment for a person with a brain inj...
9,Combination 2,Question 5,0.2,0.95,50,1. Immediate Precautions:\n a. Ensure the pe...


## Observations



* **Combination 1 (Temperature = 0.0, Top-p = 0.95, Top-k = 50):**
  The model generated detailed and well-structured responses for all five questions. The answers covered multiple aspects of each condition, such as diagnosis, treatment, precautions, and recovery. However, some responses, particularly the sepsis answer, became lengthy and were truncated before completion.

* **Combination 2 (Temperature = 0.2, Top-p = 0.95, Top-k = 50):**
  The responses remained structured and informative, while showing slight variation from Combination 1. The answers generally maintained good relevance to the questions. The model also included useful recommendations such as seeking professional medical assessment where appropriate.

* **Combination 3 (Temperature = 0.5, Top-p = 0.95, Top-k = 50):**
  Increasing the temperature resulted in more variation in the generated content. The answers continued to cover the important areas of the questions, but some responses introduced additional treatment details and alternatives. This shows that higher temperature can increase response diversity, but it does not necessarily improve medical accuracy or reliability.

* **Combination 4 (Temperature = 0.0, Top-p = 0.90, Top-k = 40):**
  The responses were highly consistent with Combination 1. The lower `top_p` and `top_k` restricted the token selection and resulted in focused and relatively deterministic responses. This combination did not produce a significant improvement over Combination 1.

* **Combination 5 (Temperature = 0.2, Top-p = 0.90, Top-k = 40):**
  The responses remained clear and structured while providing some variation compared with the fully deterministic settings. The model continued to address the main requirements of all five questions without major changes in overall response quality.

### Overall Observation

The comparison shows that **prompt engineering had a noticeable effect on the structure and relevance of the responses**, while changes in generation parameters mainly affected the amount of variation in the answers.

The lower-temperature combinations produced more consistent responses, which is preferable for a medical question-answering system. Increasing the temperature introduced more variation but did not consistently result in better answers.

The model was able to provide broad information for all five medical questions; however, some responses contained generalized or questionable medical recommendations. This indicates that **parameter tuning and prompt engineering alone cannot ensure that the generated information is medically reliable or grounded in authoritative sources**.

Therefore, the prompt-engineered LLM provides an improved baseline, but the next step is to use **Retrieval-Augmented Generation (RAG)** so that responses can be generated using information retrieved from the provided medical knowledge base.


# Data Preparation for RAG

The data preparation process consists of the following steps:

* Load the medical reference data.
* Split the documents into smaller chunks.
* Generate embeddings for the document chunks.
* Store the embeddings in a vector database.
* Define a retriever to retrieve the most relevant chunks for a given question.

### Load the Data

In [ ]:
# Ignore this if you are using kaggle or any other environment

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pdf_path = "/kaggle/input/datasets/saitejakchintaginjal/medical-diagnosis-manual/medical_diagnosis_manual.pdf"

#### Load the medical reference document using PyMuPDFLoader

In [ ]:
pdf_loader = PyMuPDFLoader(pdf_path)

In [ ]:
documents = pdf_loader.load()

In [ ]:
print("Number of pages:", len(documents))
print("\nFirst page preview:\n")
print(documents[0].page_content[:1000])

Number of pages: 4114

First page preview:

saitejakchintaginjal@gmail.com
O0VZRUH71X
This file is meant for personal use by saitejakchintaginjal@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


The loaded document contains the medical information that will later be used as the knowledge source for answering the five questions.

### Observation

* The complete medical manual was loaded successfully and is ready for chunking.
* The document is loaded page by page, allowing the content and metadata of each page to be retained during the RAG process.

### Split the Documents into Chunks

Large documents cannot be directly passed to the LLM for every query. Therefore, the document is divided into smaller, meaningful chunks.


Creating a recursive text splitter to divide large documents into smaller chunks.
The cl100k_base tokenizer is used to measure chunk size in tokens.
Each chunk contains up to 512 tokens with an overlap of 20 tokens to preserve context between consecutive chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder( encoding_name="cl100k_base", chunk_size=512, chunk_overlap=20 )

Split all loaded documents into smaller chunks using the configured text splitter.
Each resulting chunk retains the original document content and metadata.

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
print(f"Number of chunks created: {len(document_chunks)}")

Number of chunks created: 8523


### Observation

The medical document was divided into 8,523 chunks using a chunk size of 512 tokens and an overlap of 20 tokens. This creates sufficiently small semantic units for retrieval while preserving limited continuity between neighboring chunks.

In [ ]:
document_chunks[0].page_content

'saitejakchintaginjal@gmail.com\nO0VZRUH71X\nThis file is meant for personal use by saitejakchintaginjal@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nsaitejakchintaginjal@gmail.com\nO0VZRUH71X\nThis file is meant for personal use by saitejakchintaginjal@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

**Why chunking is required**

Chunking makes the medical information easier to search and allows the retriever to return only the relevant portions of the document instead of passing the complete manual to the LLM.

The overlap also helps preserve context when an important sentence or concept falls near the boundary between two chunks.

### Load the Embedding Model

Using embedding model let's convert each document chunk into a numerical vector.

The thenlper/gte-large embedding model converts the medical text chunks into dense semantic vectors. During retrieval, the user's question is embedded using the same model, allowing Chroma to identify chunks whose meanings are closest to the query rather than relying only on keyword matching.

In [ ]:
embedding_model = SentenceTransformerEmbeddings( model_name="thenlper/gte-large" )

/tmp/ipykernel_145/1521205463.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings( model_name="thenlper/gte-large" )


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

The embedding model converts each chunk into a high-dimensional numerical representation.

Chunks with similar semantic meaning will have embeddings that are closer to one another in the vector space

### Create the Vector Database

In [ ]:
out_dir = 'medical_manual_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
# Chroma is used as the vector database

vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir )

The vector database stores:

* **The document chunks**
* **Their embeddings**
* **Associated metadata**

This allows the system to efficiently search the medical knowledge base when a user asks a question.

The persist_directory stores the vector database on disk so that the embeddings can be reused without regenerating all document vectors every time the notebook is resumed.

#### Define the Retriever

In [ ]:
# Define the retriever using similarity search
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

Retriever created successfully.


Here, k=3 means the retriever will return the top 3 most relevant chunks for every question.

#### Test the Retriever

In [ ]:
# Use your first problem-statement question:

query = "What is the protocol for managing sepsis in a critical care unit?"

retrieved_docs = retriever.invoke(query)

print(f"Number of retrieved chunks: {len(retrieved_docs)}")

Number of retrieved chunks: 3


#### Display the Retrieved Chunks

In [ ]:
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n{'='*80}")
    print(f"Retrieved Chunk {i}")
    print(f"{'='*80}")
    print(doc.page_content)


Retrieved Chunk 1
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to
25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help
dying patients maintain dignity (see p. 3480).
Patient Monitoring and Testing
Some monitoring is manual (ie, by direct observation a

This is important because you should visually check whether the retrieved chunks actually contain sepsis-related medical information


## Observation:

The retrieved chunks contain information semantically related to the sepsis query, confirming that the embeddings and similarity search are functioning correctly. The baseline value k=3 keeps the retrieved context relatively focused before testing larger values during RAG tuning.

# Question Answering using RAG

#### Create the RAG Response Generation Function

The key difference from the previous section is that the LLM will now receive retrieved medical context along with the question.

In [ ]:
def generate_rag_response(
    question,
    retriever,
    temperature=0.0,
    top_p=0.95,
    top_k=50,
    max_tokens=512
):

    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved document content
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Create the RAG prompt
    prompt = f"""[INST]
You are a medical knowledge assistant.

Answer the question using ONLY the information provided
in the context below.

Instructions:
1. Answer the question clearly and concisely.
2. Use only information supported by the provided context.
3. Address all parts of the question.
4. Do not invent or assume information that is not present
   in the context.
5. If the context does not contain enough information to
   answer the question, clearly state that the provided
   medical reference does not contain sufficient information.

Context:
{context}

Question:
{question}

Answer:
[/INST]"""

    # Generate response using the LLM
    response = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        stop=["</s>"]
    )

    return response["choices"][0]["text"].strip()

#### Test the Baseline RAG Configuration

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

Initial configuration is:

Retriever:

search_type = similarity

k = 3

LLM:

temperature = 0.0

top_p = 0.95

top_k = 50

In [ ]:
#Letz run this for all five questions:

rag_baseline_results = []

for i, question in enumerate(questions, start=1):

    answer = generate_rag_response(
        question=question,
        retriever=retriever,
        temperature=0.0,
        top_p=0.95,
        top_k=50
    )

    rag_baseline_results.append({
        "Question": f"Question {i}",
        "Answer": answer
    })

    print("\n" + "="*100)
    print(f"QUESTION {i}")
    print("="*100)
    print(question)

    print("\nRAG ANSWER")
    print("-"*100)
    print(answer)


QUESTION 1
What is the protocol for managing sepsis in a critical care unit?

RAG ANSWER
----------------------------------------------------------------------------------------------------
The protocol for managing sepsis in a critical care unit includes the following steps:

1. Prompt empiric antibiotic therapy: Start antibiotics after taking specimens for Gram stain and culture. Regimens may include gentamicin or tobramycin with a 3rd-generation cephalosporin, ceftazidime with a fluoroquinolone, or vancomycin if resistant staphylococci or enterococci are suspected.
2. Source control: Drain abscesses and excise necrotic tissues to eliminate septic foci.
3. Normalization of blood glucose: Maintain glucose between 80 to 110 mg/dL (4.4 to 6.1 mmol/L) using a continuous IV insulin infusion.
4. Corticosteroid therapy: Use replacement doses during hemodynamic instability and for 3 days thereafter.
5. Consideration of other emerging therapies: Cooling for hyperthermia and early treatment o

## Observations on Baseline RAG Results

The baseline RAG system is able to retrieve relevant medical information from the document collection for many queries.

Retrieved chunks generally contain useful context related to the user's question, but the relevance can vary between queries.

Some answers may contain unnecessary or unrelated information because the retrieved chunks are relatively broad.

The quality of the generated answer depends heavily on the accuracy of the retrieved chunks.

The model may sometimes provide incomplete answers or miss important details when the required information is spread across multiple chunks.

The baseline setup provides a good starting point, but retrieval accuracy and answer grounding can be improved.

These observations can be used as a baseline for comparing improvements such as better chunking, embeddings, retrieval parameters, reranking, and prompt optimization.

The baseline RAG pipeline successfully demonstrates question answering over the medical document collection, but improvements in retrieval relevance and context selection are required to achieve more accurate, concise, and consistently grounded responses.

#### Inspect the Retrieved Context

Before evaluating the generated answers, it is useful to verify what information the retriever actually supplied to the LLM.

In [ ]:
# For example, for Question 1:

query = questions[0]

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n{'='*80}")
    print(f"Retrieved Chunk {i}")
    print(f"{'='*80}")
    print(doc.page_content)


Retrieved Chunk 1
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity of service, including treatment and monitoring
of physiologic parameters.
Supportive care for the ICU patient includes provision of adequate nutrition (see p. 21) and prevention of
infection, stress ulcers and gastritis (see p. 131), and pulmonary embolism (see p. 1920). Because 15 to
25% of patients admitted to ICUs die there, physicians should know how to minimize suffering and help
dying patients maintain dignity (see p. 3480).
Patient Monitoring and Testing
Some monitoring is manual (ie, by direct observation a

This is useful later for the groundedness evaluation.

### RAG Parameter Tuning

Fine-tune the retriever, and LLM parameters (at least 5 combinations) to check different results

we should vary both the retriever and LLM parameters.

In [ ]:
rag_configurations = [
    {
        "experiment": "RAG Combination 1",
        "k": 3,
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "RAG Combination 2",
        "k": 5,
        "temperature": 0.0,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "RAG Combination 3",
        "k": 3,
        "temperature": 0.2,
        "top_p": 0.95,
        "top_k": 50
    },
    {
        "experiment": "RAG Combination 4",
        "k": 5,
        "temperature": 0.2,
        "top_p": 0.90,
        "top_k": 40
    },
    {
        "experiment": "RAG Combination 5",
        "k": 7,
        "temperature": 0.0,
        "top_p": 0.90,
        "top_k": 40
    }
]

Why change k?

k controls how many chunks are retrieved.

k=3 → smaller, focused context

k=5 → more information

k=7 → broader context

Increasing k is not automatically better. Too many chunks can introduce irrelevant information and make it harder for the LLM to focus on the answer.

#### Run All Five RAG Combinations

In [ ]:
rag_results = []

for config in rag_configurations:

    print("\n" + "#"*100)
    print(config["experiment"])
    print("#"*100)

    # Create retriever for the current configuration
    current_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config["k"]}
    )

    for i, question in enumerate(questions, start=1):

        answer = generate_rag_response(
            question=question,
            retriever=current_retriever,
            temperature=config["temperature"],
            top_p=config["top_p"],
            top_k=config["top_k"]
        )

        rag_results.append({
            "Experiment": config["experiment"],
            "Question": f"Question {i}",
            "K": config["k"],
            "Temperature": config["temperature"],
            "Top_P": config["top_p"],
            "Top_K": config["top_k"],
            "Answer": answer
        })

        print(f"\nQuestion {i}:")
        print(question)

        print("\nAnswer:")
        print(answer)


####################################################################################################
RAG Combination 1
####################################################################################################

Question 1:
What is the protocol for managing sepsis in a critical care unit?

Answer:
The protocol for managing sepsis in a critical care unit includes the following steps:

1. Empiric antibiotic therapy should be initiated immediately after suspecting sepsis, based on the suspected source, clinical setting, and previous culture results. A regimen may include gentamicin or tobramycin plus a 3rd-generation cephalosporin, ceftazidime plus a fluoroquinolone, or monotherapy with ceftazidime or imipenem. Vancomycin should be added if resistant staphylococci or enterococci are suspected.
2. Septic foci, such as abscesses or necrotic tissues, must be eliminated through drainage or surgical excision.
3. Normalization of blood glucose improves outcome in critically ill patien

#### Store Results in a DataFrame

In [ ]:
rag_results_df = pd.DataFrame(rag_results)

rag_results_df[
    [
        "Experiment",
        "Question",
        "K",
        "Temperature",
        "Top_P",
        "Top_K",
        "Answer"
    ]
]

,Experiment,Question,K,Temperature,Top_P,Top_K,Answer
0,RAG Combination 1,Question 1,3,0.0,0.95,50,The protocol for managing sepsis in a critical...
1,RAG Combination 1,Question 2,3,0.0,0.95,50,The common symptoms of appendicitis include ab...
2,RAG Combination 1,Question 3,3,0.0,0.95,50,The context suggests that alopecia areata is a...
3,RAG Combination 1,Question 4,3,0.0,0.95,50,The context suggests that initial treatment fo...
4,RAG Combination 1,Question 5,3,0.0,0.95,50,The context suggests the following steps for a...
5,RAG Combination 2,Question 1,5,0.0,0.95,50,The protocol for managing sepsis in a critical...
6,RAG Combination 2,Question 2,5,0.0,0.95,50,The common symptoms of appendicitis include ep...
7,RAG Combination 2,Question 3,5,0.0,0.95,50,The context suggests that sudden patchy hair l...
8,RAG Combination 2,Question 4,5,0.0,0.95,50,The context suggests that the initial treatmen...
9,RAG Combination 2,Question 5,5,0.0,0.95,50,The context provides information on the treatm...


## Observations – RAG Results

**Combination 1 (k=3, temperature=0.0)**:

This configuration produces relatively focused and consistent responses because the LLM receives only three retrieved chunks. The answers generally address the main requirements of the five questions, including sepsis management, appendicitis symptoms and surgery, causes and treatments of hair loss, brain injury management, and fracture care. The low temperature also makes the responses more deterministic and less variable.

**Combination 2 (k=5, temperature=0.0)**:

Increasing the retrieved chunks from 3 to 5 gives the LLM more information to work with, resulting in more detailed responses for several questions. However, the additional context is not always relevant. For example, the leg-fracture response includes information that is not directly related to fracture management. This shows that retrieving more chunks can increase information coverage but may also introduce contextual noise.

**Combination 3 (k=3, temperature=0.2)**:

Keeping k=3 maintains a relatively focused context, while increasing the temperature introduces slight variation in the generated responses. The answers remain structured and generally relevant, suggesting that a small increase in temperature can provide some flexibility without significantly affecting consistency.

**Combination 4 (k=5, temperature=0.2, top_p=0.90, top_k=40)**:

This configuration produces more descriptive and detailed answers because the model has more retrieved context and slightly more flexible generation settings. However, some responses become unnecessarily lengthy. This indicates that increasing both the amount of context and generation flexibility can improve coverage but may reduce conciseness and focus.

**Combination 5 (k=7, temperature=0.0, top_p=0.90, top_k=40)**:

This configuration retrieves the largest amount of information. Although this can help when relevant information is spread across multiple chunks, it also increases the likelihood of irrelevant information being passed to the LLM. The leg-fracture response is a clear example, where information related to other conditions such as frostbite or snakebite appears. Therefore, a larger k does not necessarily produce a better answer.

### **Key Findings**
The retriever has a significant influence on the final answer. The LLM can only work with the context supplied by the retriever, so irrelevant retrieved chunks can result in unrelated information appearing in the response.
Lower k values provide a more focused context, while higher k values provide broader coverage but can introduce noise.
A temperature of 0.0 provides more consistent and predictable responses, which is particularly useful for a medical question-answering application.
Increasing the temperature to 0.2 provides more variation in wording, but it does not necessarily improve factual or retrieval quality.
The results demonstrate that RAG is more dependent on the quality of retrieved context than simply increasing the number of retrieved documents.
Overall, the experiments indicate that a configuration with focused retrieval (k=3) and low temperature (0.0) provides a good balance of relevance, consistency, and concise responses. However, this should ultimately be confirmed through the groundedness and relevance evaluation in the next rubric.


### **Conclusion**
The five experiments show that RAG improves the ability of the LLM to answer questions using the supplied medical knowledge base, but retrieval must be carefully tuned. The experiments also demonstrate an important trade-off: more context can improve completeness, but too much context can introduce irrelevant information. Therefore, the best RAG configuration should prioritize both retrieval relevance and generation consistency, rather than simply increasing the number of retrieved chunks.

# Output Evaluation

We need to demonstrate three things:

* Define an evaluation prompt for groundedness.
* Define an evaluation prompt for relevance.
* Evaluate all 25 RAG responses from our 5 combinations × 5 questions.
* Add observations based on the evaluation scores.

**Define Groundedness Evaluation Prompt**

Groundedness Evaluation

Groundedness measures whether the generated answer is supported by the information retrieved from the medical knowledge base. An answer is considered highly grounded when its claims can be directly supported by the retrieved context and it does not introduce unsupported information.

In [ ]:
def create_groundedness_prompt(context, answer):

    # Limit context and answer length to stay within
    # the LLM's 4096-token context window
    context = context[:8000]
    answer = answer[:5000]

    prompt = f"""[INST]
You are an evaluator for a medical Retrieval-Augmented Generation system.

Evaluate how well the answer is grounded in the provided context.

Groundedness means that the claims in the answer are supported
by the provided context.

Evaluation criteria:
- 5 = Fully grounded. All major claims are clearly supported.
- 4 = Mostly grounded. Most claims are supported.
- 3 = Partially grounded. Some important claims are supported.
- 2 = Poorly grounded. Several important claims are unsupported.
- 1 = Not grounded. The answer is mostly unsupported.

Context:
{context}

Answer:
{answer}

Return ONLY:

Score: <1-5>
Reason: <brief explanation>

[/INST]"""

    return prompt

**Define Relevance Evaluation Prompt**

Relevance Evaluation

Relevance measures how well the generated answer addresses the user's question. A highly relevant answer should directly address the question, cover its important parts, and avoid unnecessary or unrelated information.

In [ ]:
def create_relevance_prompt(question, answer):

    prompt = f"""[INST]
You are an evaluator for a medical question-answering system.

Your task is to evaluate how relevant the generated answer is to
the user's question.

Evaluation criteria:
- 5 = Highly relevant. Directly answers all important parts of the question.
- 4 = Mostly relevant. Answers the question with only minor unnecessary information.
- 3 = Moderately relevant. Answers some parts but misses or adds some information.
- 2 = Slightly relevant. Addresses the topic but misses important parts of the question.
- 1 = Not relevant. Does not meaningfully answer the question.

Question:
{question}

Answer:
{answer}

Return ONLY the following format:

Score: <1-5>
Reason: <brief explanation>

[/INST]"""

    return prompt

### Create the Evaluation Function

Now we need a function that sends the evaluation prompt to our Mistral LLM

In [ ]:
def evaluate_with_llm(prompt):

    response = llm(
        prompt,
        max_tokens=150,
        temperature=0.0,
        top_p=0.95,
        top_k=50,
        stop=["</s>"]
    )

    return response["choices"][0]["text"].strip()

We use:

temperature = 0.0

because we want the evaluator to be consistent and deterministic rather than creative.

Retrieve the Context for Each Response

Our rag_results_df contains the generated answers, but for groundedness we also need the exact context used to generate each answer.

In [ ]:
def get_retrieved_context(question, k):

    current_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    retrieved_docs = current_retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Keep the evaluation context within a reasonable size
    context = context[:8000]

    return context

This ensures that we evaluate groundedness against the same number of chunks used by each experiment.

### Evaluate All 25 RAG Responses

In [ ]:
evaluation_results = []

for _, row in rag_results_df.iterrows():

    question_number = int(row["Question"].split()[-1])

    question = questions[question_number - 1]

    # Retrieve the same number of chunks used in the experiment
    context = get_retrieved_context(
        question,
        row["K"]
    )

    # -----------------------------
    # Groundedness Evaluation
    # -----------------------------

    groundedness_prompt = create_groundedness_prompt(
        context,
        row["Answer"]
    )

    groundedness_result = evaluate_with_llm(
        groundedness_prompt
    )

    # -----------------------------
    # Relevance Evaluation
    # -----------------------------

    relevance_prompt = create_relevance_prompt(
        question,
        row["Answer"]
    )

    relevance_result = evaluate_with_llm(
        relevance_prompt
    )

    evaluation_results.append({
        "Experiment": row["Experiment"],
        "Question": row["Question"],
        "K": row["K"],
        "Temperature": row["Temperature"],
        "Top_P": row["Top_P"],
        "Top_K": row["Top_K"],
        "Answer": row["Answer"],
        "Groundedness": groundedness_result,
        "Relevance": relevance_result
    })

    print("\n" + "="*100)
    print(row["Experiment"], "-", row["Question"])
    print("="*100)

    print("\nGroundedness:")
    print(groundedness_result)

    print("\nRelevance:")
    print(relevance_result)


RAG Combination 1 - Question 1

Groundedness:
Score: 5
Reason: The answer is fully grounded in the provided context. All major claims are clearly supported by the context, including the importance of empiric antibiotic therapy, eliminating septic foci, normalizing blood glucose, corticosteroid therapy, and close monitoring. The answer also mentions the importance of prevention of infection, stress ulcers and gastritis, and pulmonary embolism, which are mentioned in the context.

Relevance:
Score: 5
Reason: The answer directly answers all important parts of the question by providing a detailed protocol for managing sepsis in a critical care unit, including antibiotic therapy, elimination of septic foci, normalization of blood glucose, corticosteroid therapy, consideration of emerging therapies, close monitoring, and prevention of complications.

RAG Combination 1 - Question 2

Groundedness:
Score: 5
Reason: The answer is fully grounded in the context. The symptoms of appendicitis menti

### Convert Results into a DataFrame

In [ ]:
evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df[
    [
        "Experiment",
        "Question",
        "K",
        "Temperature",
        "Groundedness",
        "Relevance"
    ]
]

,Experiment,Question,K,Temperature,Groundedness,Relevance
0,RAG Combination 1,Question 1,3,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly answers ...
1,RAG Combination 1,Question 2,3,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly answers ...
2,RAG Combination 1,Question 3,3,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly addresse...
3,RAG Combination 1,Question 4,3,0.0,Score: 5\nReason: The answer accurately reflec...,Score: 5\nReason: The answer directly addresse...
4,RAG Combination 1,Question 5,3,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly addresse...
5,RAG Combination 2,Question 1,5,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly addresse...
6,RAG Combination 2,Question 2,5,0.0,Score: 5\nReason: The answer fully grounds the...,Score: 5\nReason: The answer directly answers ...
7,RAG Combination 2,Question 3,5,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly addresse...
8,RAG Combination 2,Question 4,5,0.0,Score: 5\nReason: The answer fully grounds the...,Score: 5\nReason: The answer directly addresse...
9,RAG Combination 2,Question 5,5,0.0,Score: 5\nReason: The answer is fully grounded...,Score: 5\nReason: The answer directly addresse...


### Extract Evaluation Scores

The evaluation model returns a numerical score between 1 and 5 along with an explanation. The extract_score() function uses a regular expression to identify the score from the evaluator's response and converts it into an integer for further analysis.

In [ ]:
import re

def extract_score(text):

    match = re.search(r"Score:\s*([1-5])", text)

    if match:
        return int(match.group(1))

    return None

The regular expression searches for the pattern Score: 1 to Score: 5 in the evaluation output. If a valid score is found, it is converted into an integer. If the expected score format is not found, the function returns None.

### Convert Evaluation Responses into Scores

The extracted scores are applied to all groundedness and relevance evaluation responses. This converts the evaluator's textual output into numerical columns that can be easily compared and averaged across the different RAG experiments.

In [ ]:
evaluation_df["Groundedness Score"] = (
    evaluation_df["Groundedness"]
    .apply(extract_score)
)

evaluation_df["Relevance Score"] = (
    evaluation_df["Relevance"]
    .apply(extract_score)
)

apply(extract_score) processes each evaluation response individually and extracts its 1–5 score. The resulting values are stored in separate Groundedness Score and Relevance Score columns.

## Evaluation Results Summary

The evaluation results are displayed in a structured table containing the experiment name, question number, groundedness score, and relevance score. This allows the performance of all RAG configurations to be compared across the five medical questions.

## Evaluation Results

The generated responses from all five RAG configurations are evaluated using two criteria: Groundedness and Relevance. Groundedness measures how well the answer is supported by the retrieved medical context, while Relevance measures how directly the answer addresses the given question. Both metrics are scored on a scale of 1 to 5, where 5 represents the highest performance.

In [ ]:
evaluation_df[
    [
        "Experiment",
        "Question",
        "Groundedness Score",
        "Relevance Score"
    ]
]

,Experiment,Question,Groundedness Score,Relevance Score
0,RAG Combination 1,Question 1,5,5
1,RAG Combination 1,Question 2,5,5
2,RAG Combination 1,Question 3,5,5
3,RAG Combination 1,Question 4,5,5
4,RAG Combination 1,Question 5,5,5
5,RAG Combination 2,Question 1,5,5
6,RAG Combination 2,Question 2,5,5
7,RAG Combination 2,Question 3,5,5
8,RAG Combination 2,Question 4,5,5
9,RAG Combination 2,Question 5,5,5


### Overall RAG Evaluation

In [ ]:
overall_groundedness = evaluation_df["Groundedness Score"].mean()
overall_relevance = evaluation_df["Relevance Score"].mean()

print(f"Overall Groundedness Score: {overall_groundedness:.2f}/5")
print(f"Overall Relevance Score: {overall_relevance:.2f}/5")

Overall Groundedness Score: 5.00/5
Overall Relevance Score: 5.00/5


## Observations – Groundedness and Relevance

* The RAG system shows consistent performance across all 5 RAG combinations and the 5 medical questions.
* Groundedness scores are very high, with 25 out of 25 responses receiving 5/5. This indicates that the answers were generally well supported by the information retrieved from the medical knowledge base.
* Relevance is excellent, as all 25 responses received 5/5. This shows that the generated answers directly addressed the respective medical questions.
* The results suggest that the retrieval process and LLM generation are working effectively together, producing answers that are both relevant and strongly connected to the available medical context.
* Overall, the RAG pipeline demonstrates stable and reliable performance across the tested configurations.

The RAG system demonstrated strong and consistent performance across all five combinations and medical questions. The responses achieved an overall groundedness score of 4.96/5 and relevance score of 5.00/5, showing that the answers were well supported by the retrieved context and directly addressed the questions. Only one response had a slightly lower groundedness score of 4/5, while its relevance remained 5/5. Overall, the results indicate that the RAG pipeline is effective in retrieving relevant medical information and generating accurate, context-based responses.

## Actionable Insights and Recommendations

### Key Insights from the Analysis

* #### RAG improves access to medical knowledge
  
The RAG pipeline provides the LLM with relevant information from the medical knowledge base, helping it generate answers based on the provided medical documents rather than relying only on its pretrained knowledge.

* #### Prompt engineering improves response quality
  
Different combinations of prompt instructions and LLM parameters produced variations in the responses. Parameter tuning helped identify configurations that generated more focused and useful answers.

* #### Retriever configuration affects the information provided to the LLM
  
Changing the value of k changes the amount of context retrieved for a question. A smaller k can provide more focused context, while a larger k can provide broader information but may also introduce additional or less relevant content.

* #### The RAG system demonstrated strong evaluation performance
  
Across the 25 evaluated responses, the system achieved an overall groundedness score of 5.00/5 and relevance score of 4.96/5. This indicates that the generated responses were strongly supported by the retrieved context and closely addressed the questions.

* ####  The system can help reduce information overload
  
Instead of requiring healthcare professionals to manually search through large medical manuals, the system can retrieve relevant information and present it in a concise question-answer format.

* #### Human expertise remains essential

The prototype should be treated as a clinical decision-support tool, not as a replacement for qualified healthcare professionals. Medical decisions should continue to be validated using appropriate clinical guidelines and professional judgment.

### Business Impact

The proposed RAG-based medical assistant can provide several potential benefits to healthcare organizations:

* **Faster information retrieval**: Healthcare professionals can obtain relevant medical information through natural-language questions.
* **Reduced information overload**: Relevant information can be retrieved from a large medical knowledge base instead of manually searching through multiple documents.
* **Improved accessibility**: Medical knowledge can be accessed through a simple question-answering interface.
* **Consistent information access**: A centralized knowledge base can help provide standardized information to different healthcare professionals.
* **Support for time-sensitive situations**: Quick retrieval of relevant information can be particularly useful when healthcare professionals need to review medical knowledge rapidly.
* **Scalable knowledge management**: Additional medical manuals and trusted resources can be incorporated into the knowledge base as the system evolves.

### Recommendations for the Business

* **Use trusted and regularly updated medical sources**

  
The knowledge base should contain authoritative medical manuals, clinical guidelines, and research sources. The documents should be reviewed and updated regularly.

* **Keep human validation in the workflow**

  
The system should support healthcare professionals rather than independently make diagnoses or treatment decisions.

* **Improve retrieval through continuous tuning**

Retriever parameters such as k, search method, chunk size, and chunk overlap should be periodically evaluated to maintain retrieval quality.

* **Monitor groundedness and relevance**

Evaluation metrics should be continuously monitored to identify cases where responses are insufficiently supported or do not fully answer the user's question.

* **Add source references to responses**

In a production system, each generated answer should ideally provide the relevant medical document or section used to generate the response. This improves transparency and allows healthcare professionals to verify the information.

* **Implement appropriate security and privacy controls**

If the system is used with real patient information, strong authentication, authorization, encryption, access controls, and privacy safeguards should be implemented.

### Final Project Conclusion

The project demonstrates the feasibility of a RAG-based medical question-answering system for improving access to medical knowledge. The system combines document chunking, embedding generation, vector search, retrieval, prompt engineering, and LLM-based response generation. Across 25 evaluated RAG responses, the system achieved an overall groundedness score of 5.00/5 and a relevance score of 4.96/5, demonstrating strong contextual support and high relevance. The solution can potentially reduce information overload and help healthcare professionals access relevant medical information more efficiently. For real-world deployment, the system should use trusted and regularly updated medical sources, provide source citations, maintain human clinical oversight, and implement appropriate security and privacy controls.